# register-back-fn-after-wrap — faded example 3: Register Three Operations and Verify Table Completeness (Faded)

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-back-fn-after-wrap`. The last cell reports your progress on the `Backprop: register back fn` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: register back fn` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`register-back-fn-after-wrap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "register-back-fn-after-wrap"
DD_SUBTOPIC = "Backprop: register back fn"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A well-populated backward-function table is the foundation of the autograd dispatcher. After calling `register_log`, `register_multiply`, and `register_neg`, the table should contain exactly four entries: one for `log` (unary), two for `multiply` (binary), and one for `neg` (unary). The `add_back_func` method stores them all under `(fwd_fn, argnum)` keys.

## Faded exercise 3

The `BackwardFuncLookup` class and all individual back functions (`log_back`, `multiply_back0`, `multiply_back1`, `neg_back`) are already defined. Your task is to **implement `register_all(BACK_FUNCS)`** which calls all three registration functions in sequence: `register_log`, `register_multiply`, and `register_neg`.

After your implementation runs, `BACK_FUNCS._table` should have exactly 4 entries.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]

def log_back(grad_out, out, x):
    return grad_out / x

def multiply_back0(grad_out, out, x, y):
    return grad_out * y

def multiply_back1(grad_out, out, x, y):
    return grad_out * x

def neg_fn(x):
    return -x

def neg_back(grad_out, out, x):
    return -grad_out

def register_log(BACK_FUNCS):
    BACK_FUNCS.add_back_func(t.log, 0, log_back)

def register_multiply(BACK_FUNCS):
    BACK_FUNCS.add_back_func(t.multiply, 0, multiply_back0)
    BACK_FUNCS.add_back_func(t.multiply, 1, multiply_back1)

def register_neg(BACK_FUNCS):
    BACK_FUNCS.add_back_func(neg_fn, 0, neg_back)

def register_all(BACK_FUNCS: BackwardFuncLookup) -> None:
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above


def _test():
    import torch as t
    BACK_FUNCS = BackwardFuncLookup()
    register_all(BACK_FUNCS)
    assert len(BACK_FUNCS._table) == 4, f'Expected 4 entries, got {len(BACK_FUNCS._table)}'
    assert BACK_FUNCS.get_back_func(t.log, 0) is log_back
    assert BACK_FUNCS.get_back_func(t.multiply, 0) is multiply_back0
    assert BACK_FUNCS.get_back_func(t.multiply, 1) is multiply_back1
    assert BACK_FUNCS.get_back_func(neg_fn, 0) is neg_back


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]

def log_back(grad_out, out, x):
    return grad_out / x

def multiply_back0(grad_out, out, x, y):
    return grad_out * y

def multiply_back1(grad_out, out, x, y):
    return grad_out * x

def neg_fn(x):
    return -x

def neg_back(grad_out, out, x):
    return -grad_out

def register_log(BACK_FUNCS):
    BACK_FUNCS.add_back_func(t.log, 0, log_back)

def register_multiply(BACK_FUNCS):
    BACK_FUNCS.add_back_func(t.multiply, 0, multiply_back0)
    BACK_FUNCS.add_back_func(t.multiply, 1, multiply_back1)

def register_neg(BACK_FUNCS):
    BACK_FUNCS.add_back_func(neg_fn, 0, neg_back)

def register_all(BACK_FUNCS: BackwardFuncLookup) -> None:
    register_log(BACK_FUNCS)
    register_multiply(BACK_FUNCS)
    register_neg(BACK_FUNCS)
```
</details>